In [1]:
import numpy as np
import math

Pasting the data:

In [2]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]
data

[[12.0, 1.5, 1, 'Wine'],
 [5.0, 2.0, 0, 'Beer'],
 [40.0, 0.0, 1, 'Whiskey'],
 [13.5, 1.2, 1, 'Wine'],
 [4.5, 1.8, 0, 'Beer'],
 [38.0, 0.1, 1, 'Whiskey'],
 [11.5, 1.7, 1, 'Wine'],
 [5.5, 2.3, 0, 'Beer']]

Step 1: Encoding the data set

In [3]:
#to convert the labels into ints 
labels = {"Wine": 0,"Beer" : 1,"Whiskey" : 2 }

#to create the np arrays of the feature and the labels 
X  = np.array( [x[0:3] for x in data ])
print(X)
print()
y = np.array([labels [x[-1]] for x in data])
print(y)

[[12.   1.5  1. ]
 [ 5.   2.   0. ]
 [40.   0.   1. ]
 [13.5  1.2  1. ]
 [ 4.5  1.8  0. ]
 [38.   0.1  1. ]
 [11.5  1.7  1. ]
 [ 5.5  2.3  0. ]]

[0 1 2 0 1 2 0 1]


Step 2 : Implement GINI Impurity

Now we need to create a GINI impurity function to calculate the GINI impurity of every iteration we try. Here what we do is that we will pass a group that would be splitted based on some feature and threshold we just need to write a function that will use the GINI inpurity formula and find what is the impurity so that we can check for the best split .

In [4]:
def GINI (list):
    values , count = np.unique(list,return_counts=True)              #if i do not write values then count[0] will contain values and count[1] will have the counts
    total = count.sum()
    probability_sq_sum =  np.sum(  [(p/total)**2   for p in count ]  )
    return 1-probability_sq_sum


Step 3 : Implement the best split finder

so the best split split finder we check for all possible feature- threshold variations and sort it then send the sorted list to the GINI to find the impurity and then select the variation with the lowest GINI impurity.

In [5]:
def best_split (X,y):
    
    GINI_values = math.inf
    best_threshold = None
    best_feature_idx = None 


    for feature_idx in range(X.shape[1]):                                           #shape[1] returns the no of columns shape = [rows , cols]
       
       
        #we need to take the thresholds as midpoints of the sorted array to because direct use of the data as threshold will cause 
        #less stability as the boundary will be more biased and based on the largest value in our data


       sorted_vals = np.unique(X[:, feature_idx])
       midpoints = [(sorted_vals[i] + sorted_vals[i+1]) / 2 for i in range(len(sorted_vals)-1)]
       for threshold in midpoints:
        mask = X[:,feature_idx] <= threshold                                        #mask stores the True or False value for the condition 
        left = y[mask]                                                              # if row 1 of y = true then that row is taken
        right = y[~mask]

        if len(left) == 0 or len(right) == 0:                                       #we can avoid the iteration if either is 0
                continue     


        GINI_of_this_threshold = (len(left)/  (X.shape[0])) * GINI(left) + (len(right)/  (X.shape[0])) *GINI(right)                           
        if GINI_of_this_threshold < GINI_values:
                GINI_values = GINI_of_this_threshold
                best_threshold = threshold
                best_feature_idx = feature_idx
    
    return best_feature_idx , best_threshold
        
print(best_split(X,y))       
            
    
       

(0, np.float64(8.5))


Step 4 : Now we need to create a classs Node that will be of two types - 1. Internal node 2.leaf node
internal will have the feature , threshold and its left andd right childs 
leaf will have the value we get as ans ie the max value 

In [6]:
class Node:
    def __init__(self, feature_idx=None, threshold=None,left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right 
        self.value = value
        
                 

step 5: Now we need to write the main recursive function (actually class as With plain functions we have to pass the tree around everywhere as a parameter) that will create the decision tree .

In [7]:
class decision_tree :
    def __init__ (self, max_depth = 10,threshold_len = 2):
        self.max_depth = max_depth
        self.root = None                                                #the root node that will be initialized by fit 
        self.threshold_len = threshold_len
    def fit (self, X, y):
        self.root = self._build_tree(X,y,depth = 0)                      #fit needs to save the tree somewhere that predict can access later. self.root stores it inside the object so any method of that object can reach it.
    #we have used depth = 0 as "depth" is the counter that will be increamented every recursion initial node will have it set = 0
    #we have kept _build_tree and not build_tree to specify that this method is not meant to be called from outside the class

    #now we need to write a method that will return the value with the max count  in the leaf node 
    def _Max_value (self,group) :
        value , count = np.unique(group, return_counts = True)
        max_count_idx = np.argmax(count)
        return value [max_count_idx]






#here we build the main recursive method that creates the decision tree by taking a node then separating it to its children based 
#on the best feature and threshold then again pass its children to further build the decision tree


    def _build_tree(self,X , y , depth ):
        if depth >= self.max_depth or len(np.unique(y))== 1 or len(y) <= self.threshold_len :
           return Node(value = self._Max_value(y))                  #ie we are passing the leaf node back
        
        #now we will find the best feature and the best threshold for splitting using our best_split 
        best_feature_idx , best_threshold = best_split(X,y)


        #now we need to separate the values based on the threshold and feature
        #first the labels is separated
        mask = X[:,best_feature_idx] <= best_threshold
        left_child_y = y[mask]
        right_child_y = y[~mask]


        #now we separate the input data matrix rows acc to the threshold
        left_child_X = X[mask]
        right_child_X = X[~mask]

        #now recursively build the tree further from the new childs 
        left_child = self._build_tree(left_child_X,left_child_y,depth+1)
        right_child = self._build_tree(right_child_X,right_child_y,depth+1)

        #return the internal node 
        return Node(best_feature_idx,best_threshold,left_child,right_child,None)
           


    def predict (self, data ):
       
        predictions= [self._traverse(p, self.root) for p in data]
        return predictions
    

    def _traverse (self, p , node):
        
        if node.value != None :
            return node.value
        
        if p[node.feature_idx] <= node.threshold:
            return self._traverse(p,node.left)
        
        else:
            return self._traverse(p,node.right)









step 6 : check by implementation 

In [8]:
model = decision_tree(max_depth=5)
model.fit(X, y)

test_data = np.array([
    [6.0,  2.1, 0],
    [39.0, 0.05, 1],
    [13.0, 1.3, 1]
])
preds = model.predict(test_data)
reverse_labels = {v: k for k, v in labels.items()}
print([reverse_labels[p] for p in preds])

['Beer', 'Whiskey', 'Wine']
